In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/AthletiQ"

print("AthletiQ Drive contents:")
for item in os.listdir(DRIVE_ROOT):
    print(item)

AthletiQ Drive contents:
soccernet_ball_yolo.zip
models


In [ ]:
import os
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/AthletiQ"

MODEL_SRC = os.path.join(DRIVE_ROOT, "models", "best.pt")
MODEL_DST = "/content/best.pt"

shutil.copy2(MODEL_SRC, MODEL_DST)

print("Model copied successfully.")
print(f"Path: {MODEL_DST}")
print(f"Size: {os.path.getsize(MODEL_DST) / (1024**2):.2f} MB")

Model copied successfully.
Path: /content/best.pt
Size: 10.33 MB


In [ ]:
import os
import shutil

DRIVE_ZIP = "/content/drive/MyDrive/AthletiQ/soccernet_ball_yolo.zip"
RUNTIME_ZIP = "/content/soccernet_ball_yolo.zip"

print("Copying dataset ZIP...")
shutil.copy2(DRIVE_ZIP, RUNTIME_ZIP)

size_gb = os.path.getsize(RUNTIME_ZIP) / (1024**3)

print("\nDataset ZIP copied successfully.")
print(f"Path: {RUNTIME_ZIP}")
print(f"Size: {size_gb:.2f} GB")

Copying dataset ZIP...

Dataset ZIP copied successfully.
Path: /content/soccernet_ball_yolo.zip
Size: 17.05 GB


In [ ]:
import os
import zipfile
import time

ZIP_PATH = "/content/soccernet_ball_yolo.zip"
EXTRACT_DIR = "/content/soccernet_ball_yolo"

print("Starting extraction...")
start = time.time()

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

elapsed = time.time() - start

print(f"\nExtraction completed in {elapsed / 60:.2f} minutes.")
print(f"Extracted to: {EXTRACT_DIR}")

print("\nTop-level contents:")
for item in os.listdir(EXTRACT_DIR):
    print(" -", item)

Starting extraction...

Extraction completed in 3.97 minutes.
Extracted to: /content/soccernet_ball_yolo

Top-level contents:
 - soccernet_ball_yolo


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.7 MB/s eta 0:00:00


In [ ]:
import os
import yaml

DATA_ROOT = "/content/soccernet_ball_yolo/soccernet_ball_yolo"
DATA_YAML = "/content/soccernet_ball.yaml"

data_config = {
    "path": DATA_ROOT,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "ball"
    }
}

with open(DATA_YAML, "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print("Created:", DATA_YAML)
print()
with open(DATA_YAML, "r") as f:
    print(f.read())

Created: /content/soccernet_ball.yaml

path: /content/soccernet_ball_yolo/soccernet_ball_yolo
train: images/train
val: images/val
test: images/test
names:
  0: ball



In [ ]:
from ultralytics import YOLO
import torch
import os

# --------------------------------------------------
# Paths
# --------------------------------------------------
MODEL_PATH = "/content/best.pt"
DATA_YAML = "/content/soccernet_ball.yaml"

# --------------------------------------------------
# Verify GPU
# --------------------------------------------------
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available. Stop training.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

# --------------------------------------------------
# Load downloaded YOLO11n SoccerNet model
# --------------------------------------------------
model = YOLO(MODEL_PATH)

print("\nModel loaded:")
print(model.names)

# --------------------------------------------------
# Training
# --------------------------------------------------
results = model.train(

    # Dataset
    data=DATA_YAML,

    # Core training
    epochs=5,
    imgsz=1280,
    batch=22,
    device=0,
    workers=4,

    # -------------------------------
    # Learning-rate scheduling
    # -------------------------------
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,

    # -------------------------------
    # Regularization
    # -------------------------------
    weight_decay=0.0005,

    # -------------------------------
    # Tiny-ball-focused augmentation
    # -------------------------------
    mosaic=1.0,
    close_mosaic=2,

    mixup=0.0,
    copy_paste=0.0,

    degrees=0.0,
    translate=0.10,
    scale=0.50,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # -------------------------------
    # Early stopping
    # -------------------------------
    patience=2,

    # -------------------------------
    # Checkpointing
    # -------------------------------
    save=True,
    save_period=1,

    # -------------------------------
    # Validation
    # -------------------------------
    val=True,

    # IMPORTANT:
    # We do NOT specify test evaluation here.
    # Test remains untouched during training.

    # -------------------------------
    # Reproducibility
    # -------------------------------
    seed=0,

    # -------------------------------
    # Output
    # -------------------------------
    project="/content/ball_training",
    name="yolo11n_soccernet_5ep_controlled",
    exist_ok=True,

    verbose=True
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB

Model loaded:
{0: 'ball'}
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=22, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/soccernet_ball.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, 

In [ ]:
from ultralytics import YOLO
import torch

# --------------------------------------------------
# Paths
# --------------------------------------------------
MODEL_PATH = "/content/ball_training/yolo11n_soccernet_5ep_controlled/weights/best.pt"
DATA_YAML = "/content/soccernet_ball.yaml"

# --------------------------------------------------
# Load best trained model
# --------------------------------------------------
model = YOLO(MODEL_PATH)

print("Model loaded:")
print(model.names)

# --------------------------------------------------
# Test evaluation
# --------------------------------------------------
test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=1280,
    batch=22,
    device=0,
    workers=4,
    plots=True,
    verbose=True
)

# --------------------------------------------------
# Print final metrics
# --------------------------------------------------
print("\n" + "=" * 60)
print("FINAL TEST RESULTS — YOLO11n")
print("=" * 60)

print(f"Precision : {test_results.box.mp:.4f}")
print(f"Recall    : {test_results.box.mr:.4f}")
print(f"mAP50     : {test_results.box.map50:.4f}")
print(f"mAP50-95  : {test_results.box.map:.4f}")

print("=" * 60)

Model loaded:
{0: 'ball'}
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 46.3±14.7 MB/s, size: 246.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/test... 12750 images, 1240 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12750/12750 449.6it/s 28.4s
val: New cache created: /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 580/580 1.8it/s 5:30
                   all      12750      11844        0.7      0.593       0.59      0.233
Speed: 3.1ms preprocess, 11.3ms inference, 0.0ms loss, 1.8ms postprocess per image
Results sav

In [ ]:
import os
import shutil
import json
import csv
from datetime import datetime

# ==================================================
# SOURCE PATHS
# ==================================================
RUN_DIR = (
    "/content/ball_training/"
    "yolo11n_soccernet_5ep_controlled"
)

WEIGHTS_DIR = os.path.join(RUN_DIR, "weights")

# ==================================================
# GOOGLE DRIVE DESTINATION
# ==================================================
DRIVE_DIR = (
    "/content/drive/MyDrive/AthletiQ/"
    "ball_detector_yolo11n_5ep"
)

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "weights"), exist_ok=True)

# ==================================================
# 1. SAVE MODEL CHECKPOINTS
# ==================================================
required_files = [
    ("weights/best.pt", "weights/best.pt"),
    ("weights/last.pt", "weights/last.pt"),
    ("args.yaml", "args.yaml"),
    ("results.csv", "results.csv"),
    ("results.png", "results.png"),
]

print("Saving training artifacts...\n")

for source_rel, dest_rel in required_files:
    source = os.path.join(RUN_DIR, source_rel)
    destination = os.path.join(DRIVE_DIR, dest_rel)

    if os.path.isfile(source):
        shutil.copy2(source, destination)
        size_mb = os.path.getsize(destination) / (1024 ** 2)
        print(f"Saved: {dest_rel} ({size_mb:.2f} MB)")
    else:
        print(f"Not available yet: {source_rel}")

# ==================================================
# 2. SAVE FINAL TEST METRICS
# Requires test_results from the evaluation cell
# ==================================================
if "test_results" in globals():

    metrics = {
        "model": "YOLO11n SoccerNet",
        "checkpoint": "weights/best.pt",
        "test_split": "test",
        "imgsz": 1280,
        "precision": float(test_results.box.mp),
        "recall": float(test_results.box.mr),
        "mAP50": float(test_results.box.map50),
        "mAP50_95": float(test_results.box.map),
        "evaluation_date": datetime.now().isoformat(timespec="seconds"),
    }

    metrics_path = os.path.join(DRIVE_DIR, "test_metrics.json")

    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=4)

    # CSV copy for easy comparison across experiments
    csv_path = os.path.join(DRIVE_DIR, "test_metrics.csv")

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=metrics.keys())
        writer.writeheader()
        writer.writerow(metrics)

    print("\nSaved: test_metrics.json")
    print("Saved: test_metrics.csv")

else:
    print(
        "\nWARNING: test_results was not found. "
        "Run the test evaluation cell first, then rerun this cell."
    )

# ==================================================
# 3. VERIFY SAVED FILES
# ==================================================
print("\n" + "=" * 55)
print("FILES SAVED IN GOOGLE DRIVE")
print("=" * 55)

for root, dirs, files in os.walk(DRIVE_DIR):
    for filename in sorted(files):
        path = os.path.join(root, filename)
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"{os.path.relpath(path, DRIVE_DIR):35s} {size_mb:8.2f} MB")

print("\nDrive folder:", DRIVE_DIR)

Saving training artifacts...

Saved: weights/best.pt (5.27 MB)
Saved: weights/last.pt (5.27 MB)
Saved: args.yaml (0.00 MB)
Saved: results.csv (0.00 MB)
Saved: results.png (0.27 MB)

Saved: test_metrics.json
Saved: test_metrics.csv

FILES SAVED IN GOOGLE DRIVE
args.yaml                               0.00 MB
results.csv                             0.00 MB
results.png                             0.27 MB
test_metrics.csv                        0.00 MB
test_metrics.json                       0.00 MB
weights/best.pt                         5.27 MB
weights/last.pt                         5.27 MB

Drive folder: /content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep
